# Duplicate & Family Management
Hashes files, groups duplicates/variants into families, generates review queue.

In [1]:
# Setup environment
%run 01_config.ipynb

import os
import hashlib
import pandas as pd

Creating output structure in: C:\SKIN CANCER v2\pipe output
  11 subdirectories ready.
Set numpy/random seeds to 42.
Optional torch seeds set.
  01_config -- FINAL SUMMARY

Frozen paths:
  DATASET_ROOT       : C:\SKIN CANCER v2\DS
  NV_DIR             : C:\SKIN CANCER v2\DS\NV
  MEL_DIR            : C:\SKIN CANCER v2\DS\MEL
  BCC_DIR            : C:\SKIN CANCER v2\DS\BCC
  GROUND_TRUTH_CSV   : C:\SKIN CANCER v2\DS\ISIC_2019_Training_GroundTruth.csv
  METADATA_CSV       : C:\SKIN CANCER v2\DS\ISIC_2019_Training_Metadata.csv
  OUTPUT_ROOT        : C:\SKIN CANCER v2\pipe output
  FINAL_DATASET_ROOT : C:\SKIN CANCER v2\final DS

Class mapping      : {'NV': 0, 'MEL': 1, 'BCC': 2}

Output folders created:
  [OK] manifests
  [OK] reconciliation_reports
  [OK] duplicate_review
  [OK] audit_reports
  [OK] visual_inspection
  [OK] splits
  [OK] training_logs
  [OK] evaluation
  [OK] explainability
  [OK] robustness
  [OK] models

TensorFlow removed          : True
torch optional (not req 01-07):

## Read Manifest (reload-safe)

In [2]:
manifest_path = os.path.join(OUTPUT_ROOT, "manifests", "authoritative_raw_manifest.csv")
if "df_reconciled" not in locals():
    print("Loading reconciled manifest from disk...")
    df_reconciled = pd.read_csv(manifest_path)
print(f"Manifest loaded: {len(df_reconciled):,} rows")

Loading reconciled manifest from disk...
Manifest loaded: 20,720 rows


## Duplicate File Hashing & Family Detection

In [3]:
def compute_file_hash(filepath):
    if not os.path.exists(filepath):
        return None
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

def build_families(df_manifest):
    print("Computing file hashes...")
    df_manifest = df_manifest.copy()
    df_manifest["file_hash"] = df_manifest["full_path"].apply(compute_file_hash)

    global_hash_counts = df_manifest["file_hash"].dropna().value_counts()
    duplicated_hashes  = set(global_hash_counts[global_hash_counts > 1].index)

    families = []
    folder_col = "folder_label" if "folder_label" in df_manifest.columns else "raw_folder_label"

    for base_id, group in df_manifest.groupby("base_id_candidate"):
        family_id          = f"FAM_{base_id}"
        has_multiple_files = len(group) > 1
        multiple_folders   = len(group[folder_col].unique()) > 1
        gt_mismatch        = len(group["gt_label"].dropna().unique()) > 1
        hashes_series      = group["file_hash"].dropna()
        has_hash_collision = hashes_series.isin(duplicated_hashes).any()
        needs_review       = has_multiple_files or multiple_folders or gt_mismatch or has_hash_collision

        families.append({
            "family_id":                family_id,
            "base_id_candidate":        base_id,
            "all_member_filenames":     "|".join(group["file_name_with_extension"].astype(str)),
            "all_member_paths":         "|".join(group["full_path"].astype(str)),
            "all_member_folder_labels": "|".join(group[folder_col].astype(str)),
            "all_member_gt_labels":     "|".join(group["gt_label"].astype(str)),
            "all_member_match_status":  "|".join(group["match_status"].astype(str)),
            "all_member_lesion_ids":    "|".join(group["lesion_id"].dropna().astype(str)) if "lesion_id" in group.columns else "",
            "all_member_hashes":        "|".join(group["file_hash"].dropna().astype(str)),
            "needs_review":             needs_review,
            "review_status":            "pending" if needs_review else "clean",
            "review_decision":          "",
            "review_notes":             ""
        })
        df_manifest.loc[group.index, "family_id"] = family_id

    n_dup_hash = len(duplicated_hashes)
    n_suffix   = int((df_manifest["recognized_suffix"].fillna("") != "").sum())
    print(f"Families built                : {len(families):,}")
    print(f"  Exact duplicate hash groups : {n_dup_hash:,}")
    print(f"  Suffix/downsampled variants : {n_suffix:,}")
    print(f"  Families needing review     : {sum(f['needs_review'] for f in families):,}")
    return df_manifest, pd.DataFrame(families)

df_manifest_w_hashes, df_families = build_families(df_reconciled)
df_families[df_families["needs_review"] == True].head(3)

Computing file hashes...
Families built                : 20,720
  Exact duplicate hash groups : 28
  Suffix/downsampled variants : 1,690
  Families needing review     : 56


,family_id,base_id_candidate,all_member_filenames,all_member_paths,all_member_folder_labels,all_member_gt_labels,all_member_match_status,all_member_lesion_ids,all_member_hashes,needs_review,review_status,review_decision,review_notes
1247,FAM_ISIC_0012127,ISIC_0012127,ISIC_0012127_downsampled.jpg,C:\SKIN CANCER v2\DS\NV\ISIC_0012127_downsampl...,NV,NV,matched_exact,,7675fa6a4beaf4a1cf77d9be3dfdc10ea525cd7532831d...,True,pending,,
1484,FAM_ISIC_0012970,ISIC_0012970,ISIC_0012970_downsampled.jpg,C:\SKIN CANCER v2\DS\NV\ISIC_0012970_downsampl...,NV,NV,matched_exact,MSK4_0011472,7675fa6a4beaf4a1cf77d9be3dfdc10ea525cd7532831d...,True,pending,,
2567,FAM_ISIC_0024366,ISIC_0024366,ISIC_0024366.jpg,C:\SKIN CANCER v2\DS\NV\ISIC_0024366.jpg,NV,NV,matched_exact,HAM_0002300,b8b37d24e294c5182a1884741c109d8b3bc58edaa993e8...,True,pending,,


## Generate Review Queue

In [4]:
def generate_review_queue(df_families):
    queue_path = os.path.join(OUTPUT_ROOT, "duplicate_review", "manual_adjudication_queue.csv")
    df_queue   = df_families[df_families["needs_review"] == True].copy()
    df_queue.to_csv(queue_path, index=False)
    print(f"Review queue saved: {len(df_queue):,} families -> {queue_path}")
    return df_queue

df_queue = generate_review_queue(df_families)

Review queue saved: 56 families -> C:\SKIN CANCER v2\pipe output\duplicate_review\manual_adjudication_queue.csv


## Apply Decisions and Generate Output Manifests

In [5]:
def apply_review_decisions(df_manifest, df_families):
    cols_to_drop = [
        "final_dataset_status", "final_exclusion_reason",
        "eligible_for_training", "eligible_for_split",
        "family_review_status", "family_review_decision",
        "remove_due_to_family_review_flag", "canonical_keep_flag"
    ]
    df_manifest = df_manifest.drop(columns=[c for c in cols_to_drop if c in df_manifest.columns])

    unresolved_fam_ids = set(df_families[df_families["needs_review"] == True]["family_id"].tolist())

    def set_exclusion_status(row):
        is_flagged = row.get("family_id", "") in unresolved_fam_ids
        return pd.Series([
            "flagged_for_review" if is_flagged else "eligible",
            "duplicate_family_pending_review" if is_flagged else "none",
            not is_flagged,
            not is_flagged
        ], index=["final_dataset_status", "final_exclusion_reason",
                  "eligible_for_training", "eligible_for_split"])

    status_df   = df_manifest.apply(set_exclusion_status, axis=1)
    df_manifest = pd.concat([df_manifest, status_df], axis=1)

    clean_path = os.path.join(OUTPUT_ROOT, "manifests", "cleaned_working_manifest.csv")
    train_path = os.path.join(OUTPUT_ROOT, "manifests", "training_eligible_manifest.csv")
    resol_path = os.path.join(OUTPUT_ROOT, "duplicate_review", "final_duplicate_resolution.csv")

    df_manifest.to_csv(clean_path, index=False)

    df_training = df_manifest[df_manifest["eligible_for_training"] == True]
    df_training.to_csv(train_path, index=False)

    df_flagged = df_manifest[df_manifest["eligible_for_training"] == False].copy()
    pd.DataFrame({
        "full_path":                df_flagged["full_path"],
        "file_name_with_extension": df_flagged["file_name_with_extension"],
        "normalized_id":            df_flagged["base_id_candidate"],
        "class_label":              df_flagged["gt_label"],
        "family_id":                df_flagged["family_id"],
        "duplicate_hash_group":     df_flagged["file_hash"],
        "final_decision":           "pending_review",
        "final_exclusion_reason":   df_flagged["final_exclusion_reason"],
        "notes":                    "Flagged for duplicate/family review"
    }).to_csv(resol_path, index=False)

    print(f"Flagged for review : {len(df_flagged):,}")
    print(f"Remaining eligible : {len(df_training):,}")
    print(f"Clean manifest     : {clean_path}")
    print(f"Training manifest  : {train_path}")
    return df_manifest

df_clean_manifest = apply_review_decisions(df_manifest_w_hashes, df_families)
df_clean_manifest.head(3)

Flagged for review : 56
Remaining eligible : 20,664
Clean manifest     : C:\SKIN CANCER v2\pipe output\manifests\cleaned_working_manifest.csv
Training manifest  : C:\SKIN CANCER v2\pipe output\manifests\training_eligible_manifest.csv


,full_path,folder_name,raw_folder_label,file_name_with_extension,file_stem_raw,base_id_candidate,recognized_suffix,extension,file_exists,file_size_bytes,...,metadata_row_found,review_status,review_decision,review_notes,file_hash,family_id,final_dataset_status,final_exclusion_reason,eligible_for_training,eligible_for_split
0,C:\SKIN CANCER v2\DS\NV\ISIC_0000000.jpg,NV,NV,ISIC_0000000.jpg,ISIC_0000000,ISIC_0000000,NaN,.jpg,True,49964,...,True,provisionally_approved,NaN,NaN,153b10f7b8b82bf80badbfdf73a544312637a1950e2dad...,FAM_ISIC_0000000,eligible,none,True,True
1,C:\SKIN CANCER v2\DS\NV\ISIC_0000001.jpg,NV,NV,ISIC_0000001.jpg,ISIC_0000001,ISIC_0000001,NaN,.jpg,True,38941,...,True,provisionally_approved,NaN,NaN,6180745ca3044c6267b58dd77ae821fca7df549c64bb65...,FAM_ISIC_0000001,eligible,none,True,True
2,C:\SKIN CANCER v2\DS\NV\ISIC_0000003.jpg,NV,NV,ISIC_0000003.jpg,ISIC_0000003,ISIC_0000003,NaN,.jpg,True,45774,...,True,provisionally_approved,NaN,NaN,e3092bd47d1955c117a18acb1e236222cae99acfbd393d...,FAM_ISIC_0000003,eligible,none,True,True


In [6]:
# Final summary
_queue_path = os.path.join(OUTPUT_ROOT, "duplicate_review", "manual_adjudication_queue.csv")
_clean_path = os.path.join(OUTPUT_ROOT, "manifests", "cleaned_working_manifest.csv")
_train_path = os.path.join(OUTPUT_ROOT, "manifests", "training_eligible_manifest.csv")
_resol_path = os.path.join(OUTPUT_ROOT, "duplicate_review", "final_duplicate_resolution.csv")

_hash_counts = df_manifest_w_hashes["file_hash"].dropna().value_counts()
_dup_hash_n  = int((_hash_counts > 1).sum())
_suffix_n    = int((df_manifest_w_hashes["recognized_suffix"].fillna("") != "").sum())

print("=" * 60)
print("  03_duplicate_review -- FINAL SUMMARY")
print("=" * 60)
print(f"\nTotal rows loaded             : {len(df_reconciled):,}")
print(f"Duplicate family rows flagged : {int((df_clean_manifest['final_dataset_status'] == 'flagged_for_review').sum()):,}")
print(f"Exact duplicate hash groups   : {_dup_hash_n:,}")
print(f"Suffix/downsampled variants   : {_suffix_n:,}")
print(f"Manual review queue families  : {len(df_queue):,}")
print(f"Remaining eligible rows       : {int(df_clean_manifest['eligible_for_training'].sum()):,}")
print(f"\nOutput file verification:")
for p in [_queue_path, _clean_path, _train_path, _resol_path]:
    exists = os.path.exists(p)
    size   = os.path.getsize(p) if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {os.path.basename(p):<45} {size:>10,} bytes")
print("=" * 60)

  03_duplicate_review -- FINAL SUMMARY

Total rows loaded             : 20,720
Duplicate family rows flagged : 56
Exact duplicate hash groups   : 28
Suffix/downsampled variants   : 1,690
Manual review queue families  : 56
Remaining eligible rows       : 20,664

Output file verification:
  [OK] manual_adjudication_queue.csv                     11,623 bytes
  [OK] cleaned_working_manifest.csv                   6,855,618 bytes
  [OK] training_eligible_manifest.csv                 6,835,207 bytes
  [OK] final_duplicate_resolution.csv                    13,689 bytes
